# Single Material And Shape: Full Numerical Sweep

Inspect every available result for one material and one shape without averaging across spatial resolution, adaptive field-step resolution, or periodic mode.

In [ ]:
from pathlib import Path
import math
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np

SINGLE_GRAIN_DIR = Path.cwd()
if not (SINGLE_GRAIN_DIR / 'utils').is_dir():
    SINGLE_GRAIN_DIR = Path('python/experiments/single_grain').resolve()
sys.path.insert(0, str(SINGLE_GRAIN_DIR))

from utils.plotting import save_figure
from utils.reporting import diagnostic_counts, material_summary, missing_combinations
from utils.results import filter_results, format_summary_table, load_results, write_summary_csv

# Configuration: change only this cell for a normal single material/shape review.
MATERIAL = 'fe16n2'
SHAPE_VARIANT = 'cube'
EXCLUDED_RESULT_DIRS = {'res_15_06', 'res_16_06'}

_env_result_root = os.environ.get('SINGLE_GRAIN_RESULTS_ROOT')
if _env_result_root:
    RESULT_ROOTS = [Path(_env_result_root)]
else:
    RESULT_ROOTS = [
        path for path in sorted(SINGLE_GRAIN_DIR.glob('res_*'))
        if path.is_dir() and path.name not in EXCLUDED_RESULT_DIRS
    ]

# None means use every available value for the selected material/shape.
PERIODIC_MODES = (False, True)
RESOLUTIONS = None
FIELD_STEP_RESOLUTIONS = None
EXPECTED_SIZES_NM = None

EXPORT = True
EXPORT_DIR = SINGLE_GRAIN_DIR / 'figures' / 'compare_single_material_shape'
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True})

## Load And Select The Dataset

In [ ]:
def is_excluded_result_path(path):
    return any(part in EXCLUDED_RESULT_DIRS for part in Path(path).parts)

records, load_errors = load_results(RESULT_ROOTS)
records = [record for record in records if not is_excluded_result_path(record.path)]

selected = filter_results(records, materials=[MATERIAL], shapes=[SHAPE_VARIANT])
selected = [record for record in selected if record.periodic in PERIODIC_MODES]
if RESOLUTIONS is not None:
    selected = [record for record in selected if record.n in set(RESOLUTIONS)]
if FIELD_STEP_RESOLUTIONS is not None:
    selected = [
        record for record in selected
        if any(np.isclose(record.adaptive_dh_min_t, dh) for dh in FIELD_STEP_RESOLUTIONS)
    ]

selected = sorted(
    selected,
    key=lambda record: (
        record.periodic,
        record.adaptive_dh_min_t,
        record.n,
        record.size_nm,
        str(record.path),
    ),
)

print(f'Loaded {len(records)} result files from:')
for root in RESULT_ROOTS:
    print(f'  {Path(root).resolve()}')
if EXCLUDED_RESULT_DIRS:
    print(f'Excluded result directories: {sorted(EXCLUDED_RESULT_DIRS)}')
if load_errors:
    print()
    print(f'Skipped {len(load_errors)} malformed result file(s):')
    for path, error in load_errors[:10]:
        print(f'  {path}: {error}')

print()
print('Metric diagnostics:', diagnostic_counts(selected))
print(f'Selected {len(selected)} result(s) for {MATERIAL} / {SHAPE_VARIANT}.')
print(f'Periodic modes: {sorted({record.periodic for record in selected})}')
print(f'Spatial resolutions n: {sorted({record.n for record in selected})}')
print(f'Field-step resolutions dh_min [T]: {sorted({record.adaptive_dh_min_t for record in selected if np.isfinite(record.adaptive_dh_min_t)})}')
print(f'Sizes [nm]: {sorted({round(record.size_nm, 9) for record in selected})}')

summary = material_summary(selected)
if summary:
    print()
    print('material | shape | mu0 Ms [T] | A0 [J/m] | K0 [MJ/m3] | results | sizes [nm] | periodic')
    print('-' * 140)
    for row in summary:
        print(f"{row['material']:<18} {row['shape_variant']:<14} {row['mu0_Ms_T']:10.4g} "
              f"{row['A0_J_per_m']:11.4g} {row['K0_MJ_per_m3']:12.4g} "
              f"{row['result_count']:8d}  {row['sizes_nm']}  {row['periodic_modes']}")

## Completeness Check

In [ ]:
if selected:
    sizes_nm = EXPECTED_SIZES_NM or sorted({record.size_nm for record in selected})
    resolutions = sorted({record.n for record in selected})
    dh_min_values = sorted({record.adaptive_dh_min_t for record in selected if np.isfinite(record.adaptive_dh_min_t)})
    missing = missing_combinations(
        selected,
        materials=[MATERIAL],
        shapes=[SHAPE_VARIANT],
        sizes_nm=sizes_nm,
        resolutions=resolutions,
        dh_min_values=dh_min_values,
        periodic_modes=PERIODIC_MODES,
    )
    print(f'Missing expected combinations: {len(missing)}')
    for item in missing[:20]:
        print(' ', item)
    if len(missing) > 20:
        print('  ...')
else:
    print('No selected results to check.')

## Result Table

In [ ]:
if selected:
    print(format_summary_table(selected))
else:
    print('No matching results were found.')

## Shape Hysteresis Curves At Every Field-Step Resolution

In [ ]:
def export_safe_label(label):
    return ''.join(char if char.isalnum() or char in {'-', '_'} else '_' for char in str(label)).strip('_')

def mode_label(periodic):
    return 'periodic' if periodic else 'non-periodic'

def periodic_linestyle(periodic):
    return '-' if periodic else '--'

def style_maps(records):
    n_values = sorted({record.n for record in records})
    dh_values = sorted({record.adaptive_dh_min_t for record in records if np.isfinite(record.adaptive_dh_min_t)})
    colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [])
    markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>']
    n_colors = {n: colors[index % len(colors)] for index, n in enumerate(n_values)}
    dh_markers = {dh_min: markers[index % len(markers)] for index, dh_min in enumerate(dh_values)}
    return n_values, dh_values, n_colors, dh_markers

def add_style_legends(fig, n_values, dh_values, n_colors, dh_markers, *, include_size_note=False):
    n_handles = [
        Line2D([0], [0], color=n_colors[n], linewidth=2.0, label=f'n={n}')
        for n in n_values
    ]
    dh_handles = [
        Line2D([0], [0], color='0.35', marker=dh_markers[dh], linestyle='None', markersize=6, label=f'dh={dh:g} T')
        for dh in dh_values
    ]
    periodic_handles = [
        Line2D([0], [0], color='0.25', linestyle='--', linewidth=2.0, label='non-periodic'),
        Line2D([0], [0], color='0.25', linestyle='-', linewidth=2.0, label='periodic'),
    ]
    legend1 = fig.legend(handles=n_handles, title='Spatial resolution', loc='center left', bbox_to_anchor=(0.82, 0.72), frameon=False)
    legend2 = fig.legend(handles=dh_handles, title='Field step', loc='center left', bbox_to_anchor=(0.82, 0.48), frameon=False)
    legend3 = fig.legend(handles=periodic_handles, title='Boundary mode', loc='center left', bbox_to_anchor=(0.82, 0.27), frameon=False)
    fig.add_artist(legend1)
    fig.add_artist(legend2)
    if include_size_note:
        fig.text(0.82, 0.08, 'Hysteresis labels show grain size.', fontsize=9, color='0.35')
    return legend3

if not selected:
    print('No matching results available for hysteresis plots.')
else:
    n_values, dh_values, _, _ = style_maps(selected)
    size_values = sorted({record.size_nm for record in selected})
    colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [])
    size_colors = {size_nm: colors[index % len(colors)] for index, size_nm in enumerate(size_values)}
    for periodic in PERIODIC_MODES:
        for dh_min in dh_values:
            dh_mode_records = [
                record for record in selected
                if record.periodic == periodic and np.isclose(record.adaptive_dh_min_t, dh_min)
            ]
            if not dh_mode_records:
                print(f'No {mode_label(periodic)} results for dh_min={dh_min:g} T.')
                continue
            ncols = min(3, len(n_values))
            nrows = math.ceil(len(n_values) / ncols)
            fig, axes = plt.subplots(
                nrows,
                ncols,
                figsize=(5.2 * ncols, 3.6 * nrows),
                squeeze=False,
                sharex=True,
                sharey=True,
            )
            plotted = False
            for ax, n in zip(axes.ravel(), n_values):
                group = sorted(
                    [record for record in dh_mode_records if record.n == n],
                    key=lambda record: (record.size_nm, str(record.path)),
                )
                if not group:
                    ax.set_visible(False)
                    continue
                for record in group:
                    ax.plot(
                        record.H_T,
                        4.0 * np.pi * 1e-7 * record.Mz_A_per_m,
                        color=size_colors[record.size_nm],
                        linestyle='-',
                        linewidth=1.3,
                        label=f'{record.size_nm:g} nm',
                    )
                ax.axhline(0.0, color='0.5', linewidth=0.7)
                ax.axvline(0.0, color='0.5', linewidth=0.7)
                ax.set(title=f'n={n}', xlabel=r'$\mu_0 H$ [T]', ylabel=r'$\mu_0 M_z$ [T]')
                ax.grid(True, linestyle=':', alpha=0.6)
                ax.legend(title='Domain size', fontsize=8)
                plotted = True
            for ax in axes.ravel()[len(n_values):]:
                ax.set_visible(False)
            fig.suptitle(
                f'{MATERIAL} / {SHAPE_VARIANT}: hysteresis, '
                f'dh_min={dh_min:g} T, {mode_label(periodic)}'
            )
            fig.tight_layout()
            if EXPORT and plotted:
                save_figure(
                    fig,
                    EXPORT_DIR / (
                        f'hysteresis_{export_safe_label(MATERIAL)}_'
                        f'{export_safe_label(SHAPE_VARIANT)}_'
                        f'{export_safe_label(mode_label(periodic))}_'
                        f'dh_{export_safe_label(f"{dh_min:g}")}.png'
                    ),
                )

## Coercivity, Remanence, Squareness, And Maximum Energy Product

In [ ]:
METRIC_SPECS = {
    'abs_Hc_T': (lambda record: abs(record.metrics.Hc_T), r'$|\mu_0 H_c|$ [T]'),
    'mu0_Mr_T': (lambda record: record.metrics.mu0_Mr_T, r'$\mu_0 M_r$ [T]'),
    'Mr_over_Ms': (lambda record: record.metrics.Mr_over_Ms, r'$M_r/M_s$ [-]'),
    'BH_max_kJ_per_m3': (lambda record: record.metrics.BH_max_kJ_per_m3, r'$(BH)_{\max}$ [kJ/m$^3$]'),
}

def plot_metric_sweep(records, *, title):
    records = list(records)
    fig, axes = plt.subplots(2, 2, figsize=(15.2, 8), sharex=True)
    if not records:
        fig.suptitle(title)
        return fig, axes
    n_values, dh_values, n_colors, dh_markers = style_maps(records)
    reference_record = records[0]
    stoner_wohlfarth_hc_t = (
        2.0 * reference_record.K0_J_per_m3 / reference_record.Ms_A_per_m
    )
    for ax, (metric_name, (value_func, ylabel)) in zip(axes.ravel(), METRIC_SPECS.items()):
        for periodic in PERIODIC_MODES:
            for n in n_values:
                for dh_min in dh_values:
                    group = sorted(
                        [
                            record for record in records
                            if record.periodic == periodic
                            and record.n == n
                            and np.isclose(record.adaptive_dh_min_t, dh_min)
                        ],
                        key=lambda record: (record.size_nm, str(record.path)),
                    )
                    if not group:
                        continue
                    ax.plot(
                        [record.size_nm for record in group],
                        [value_func(record) for record in group],
                        color=n_colors[n],
                        marker=dh_markers[dh_min],
                        linestyle=periodic_linestyle(periodic),
                        linewidth=1.35,
                        markersize=4.5,
                        alpha=0.9 if periodic else 0.72,
                    )
        if metric_name == 'abs_Hc_T':
            ax.axhline(
                stoner_wohlfarth_hc_t,
                color='0.5',
                linestyle='--',
                linewidth=1.5,
                label=f'SW: {stoner_wohlfarth_hc_t:.2f}T',
            )
            ax.legend(fontsize=9, loc='best')
        ax.set(title=ylabel, xlabel='grain size [nm]', ylabel=ylabel)
        ax.grid(True, linestyle=':', alpha=0.6)
    fig.suptitle(title)
    fig.tight_layout(rect=(0.0, 0.0, 0.80, 0.95))
    add_style_legends(fig, n_values, dh_values, n_colors, dh_markers)
    return fig, axes

if selected:
    metrics_fig, metrics_axes = plot_metric_sweep(
        selected,
        title=(
            f'{MATERIAL} / {SHAPE_VARIANT}: permanent-magnet metrics '
            'for all n, dh_min, and periodic modes'
        ),
    )
    if EXPORT:
        save_figure(
            metrics_fig,
            EXPORT_DIR / f'metrics_all_{export_safe_label(MATERIAL)}_{export_safe_label(SHAPE_VARIANT)}.png',
        )
else:
    print('No matching results available for metric plots.')

## Metrics Split By Periodic Mode

In [ ]:
if selected:
    for periodic in PERIODIC_MODES:
        mode_records = [record for record in selected if record.periodic == periodic]
        if not mode_records:
            print(f'No {mode_label(periodic)} results available for metric plots.')
            continue
        fig, axes = plot_metric_sweep(
            mode_records,
            title=(
                f'{MATERIAL} / {SHAPE_VARIANT}: permanent-magnet metrics '
                f'for all n and dh_min ({mode_label(periodic)})'
            ),
        )
        if EXPORT:
            save_figure(
                fig,
                EXPORT_DIR / (
                    f'metrics_{export_safe_label(MATERIAL)}_'
                    f'{export_safe_label(SHAPE_VARIANT)}_'
                    f'{export_safe_label(mode_label(periodic))}.png'
                ),
            )
else:
    print('No matching results available for split metric plots.')

## Optional Export

In [ ]:
if EXPORT:
    csv_path = write_summary_csv(
        selected,
        EXPORT_DIR / f'{export_safe_label(MATERIAL)}_{export_safe_label(SHAPE_VARIANT)}_full_sweep.csv',
    )
    print(f'Exported figures and table to {EXPORT_DIR.resolve()}')
    print(f'CSV: {csv_path}')
else:
    print('EXPORT=False: no files were written. Set EXPORT=True in the configuration cell to save outputs.')